# Circuit Motif Discovery — Full Colab Pipeline

Run all steps from environment setup to final figures.

In [ ]:
# Cell 1: Setup (clone-first)
!nvidia-smi

%cd /content
!rm -rf /content/circuit-motif-discovery
!git clone https://github.com/mark-znidar/circuit-motif-discovery.git
%cd /content/circuit-motif-discovery
!bash setup_colab.sh

# Hugging Face auth for gated Gemma model (replace hf_xxx)
HF_TOKEN = "hf_xxx"
if HF_TOKEN != "hf_xxx":
    import os
    from huggingface_hub import login
    login(HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token configured for this runtime.")
else:
    print("Set HF_TOKEN in this cell before running graph generation.")

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# If clone fails (private/auth issue), fallback to Drive upload:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/circuit-motif-discovery
# !bash setup_colab.sh

In [ ]:
# Cell 2: One-command smoke test (~10 min)
!python scripts/run_full_pipeline.py --quick --hf_token "$HF_TOKEN"

In [ ]:
# Cell 3: Tiny graph generation (fast debug)
!python scripts/01_generate_graphs.py \
  --prompt_file prompts/prompt_corpus.json \
  --output_dir data/graphs_tiny \
  --families arithmetic \
  --max_prompts_per_family 2 \
  --hf_token "$HF_TOKEN"

import json
print(json.load(open('data/graphs_tiny/summary.json')))

In [ ]:
# Cell 4: Convert tiny graphs
!python scripts/02_convert_to_pyg.py \
  --input_dir data/graphs_tiny \
  --output_path data/circuit_dataset_tiny.pt

import torch
dataset = torch.load('data/circuit_dataset_tiny.pt', weights_only=False)
print('Num graphs:', len(dataset))

In [ ]:
# Cell 5: Tiny training
!python scripts/03_train_contrastive.py \
  --dataset_path data/circuit_dataset_tiny.pt \
  --output_dir checkpoints_tiny \
  --epochs 3 \
  --batch_size 2

import torch
ckpt = torch.load('checkpoints_tiny/best.pt', map_location='cpu', weights_only=False)
print('Best epoch:', ckpt.get('epoch'), '| Best loss:', ckpt.get('loss'))

In [ ]:
# Cell 6: Tiny evaluation
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['evaluation']['kmeans_k'] = 2
cfg['evaluation']['linear_probe_cv'] = 2
cfg['evaluation']['knn_k'] = 1
cfg['evaluation']['retrieval_k'] = 2
with open('configs/tiny_eval.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

!python scripts/04_evaluate.py \
  --dataset_path data/circuit_dataset_tiny.pt \
  --checkpoint checkpoints_tiny/best.pt \
  --output_dir results_tiny \
  --config configs/tiny_eval.yaml

from IPython.display import Image, display
for p in ['results_tiny/demo1_umap.png', 'results_tiny/demo4_metrics.png']:
    display(Image(p))

In [ ]:
# Cell 7: Save tiny results
from pathlib import Path

drive_out = Path('/content/drive/MyDrive/circuit-motif-discovery-results-tiny')
drive_out.mkdir(parents=True, exist_ok=True)
!cp -r results_tiny "{drive_out}"

print('Artifacts in results_tiny/:')
for p in sorted(Path('results_tiny').glob('*')):
    print('-', p)
print('\nCopied to:', drive_out)